In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
df = pd.read_csv('Bank Customer Churn Prediction.csv')
df = df.drop('customer_id', axis=1)
colum = df[['country', 'gender']]
le = LabelEncoder()
for col in colum:
    df[col] = le.fit_transform(df[col])
# scaler = StandardScaler()
# df[['balance']] = scaler.fit_transform(df[['balance']])
x = df.drop(['churn', 'credit_score'], axis=1)
y = df['churn']
scaler = StandardScaler()
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25)
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)
model_list = {'Rf': RandomForestClassifier(random_state=42, class_weight='balanced').fit(x_train_scaled, y_train),
'Lr': LogisticRegression(random_state=42, class_weight='balanced').fit(x_train_scaled, y_train),
'XGB': XGBClassifier(random_state=42).fit(x_train_scaled, y_train),
'DC' :DecisionTreeClassifier(random_state=42).fit(x_train_scaled, y_train)}
df


,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
0,619,0,0,42,2,0.00,1,1,1,101348.88,1
1,608,2,0,41,1,83807.86,1,0,1,112542.58,0
2,502,0,0,42,8,159660.80,3,1,0,113931.57,1
3,699,0,0,39,1,0.00,2,0,0,93826.63,0
4,850,2,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,0,1,39,5,0.00,2,1,0,96270.64,0
9996,516,0,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,0,0,36,7,0.00,1,0,1,42085.58,1
9998,772,1,1,42,3,75075.31,2,1,0,92888.52,1


In [2]:
def model_f1(model_list):
    for name, model in model_list.items():
        model_pred = model.predict(x_test_scaled)
        print(f'f1 {name}: {f1_score(y_test, model_pred)}')
model_f1(model_list)


f1 Rf: 0.6062378167641326
f1 Lr: 0.4755341144038594
f1 XGB: 0.576536312849162
f1 DC: 0.52


In [3]:
def model_accuracy(model_list):
    for name, model in model_list.items():
        model_pred = model.predict(x_test_scaled)
        print(f'accuracy {name}: {accuracy_score(y_test, model_pred)}')

model_accuracy(model_list)

accuracy Rf: 0.8384
accuracy Lr: 0.6956
accuracy XGB: 0.8484
accuracy DC: 0.7984


In [4]:
xgb = XGBClassifier(random_state=42)
param_grid_xgb = {'max_depth': [3, 5, 7], 
                  'n_estimators': [200, 300],                             
                  'learning_rate': [0.05, 0.1],      
                  'min_child_weight': [1, 3],         
                  'subsample': [0.8],          
                  'colsample_bytree': [0.8],   
                  'scale_pos_weight': [3] }                
grid = GridSearchCV(estimator=xgb, param_grid=param_grid_xgb, cv=5, scoring='f1').fit(x_train_scaled, y_train)

best_model = grid.best_estimator_
xgb_pred = best_model.predict(x_test_scaled)
print(f'f1_xgb: {f1_score(y_test, xgb_pred)}')

f1_xgb: 0.6408268733850129


In [6]:
rf = RandomForestClassifier(random_state=42)
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'class_weight': ['balanced', 'balanced_subsample', None]
}
grid = GridSearchCV(estimator=rf, param_grid=param_grid_rf, cv=5, scoring='f1', n_jobs=-1).fit(x_train_scaled, y_train)
best_model_rf = grid.best_estimator_
rf_pred = best_model_rf.predict(x_test_scaled)
print(f'f1_rf: {f1_score(y_test, rf_pred)}')


f1_rf: 0.6228622862286228


In [35]:
y_prob = best_model.predict_proba(x_test_scaled)[:,1]
threshold = 0.6
xgb_pred_new = (y_prob >= threshold).astype(int)
print(f'f1_xgb: {f1_score(y_test, xgb_pred_new)}')

f1_xgb: 0.6452905811623246


In [34]:
y_prob = best_model_rf.predict_proba(x_test_scaled)[:,1]
threshold = 0.55
rf_pred_new = (y_prob >= threshold).astype(int)
print(f'f1_rf: {f1_score(y_test, rf_pred_new)}')

f1_rf: 0.6316793893129771


In [7]:
from sklearn.metrics import classification_report

print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.91      0.88      0.89      1977
           1       0.59      0.66      0.62       523

    accuracy                           0.83      2500
   macro avg       0.75      0.77      0.76      2500
weighted avg       0.84      0.83      0.84      2500



In [8]:
from sklearn.metrics import classification_report

print(classification_report(y_test, xgb_pred))

              precision    recall  f1-score   support

           0       0.92      0.87      0.89      1977
           1       0.58      0.71      0.64       523

    accuracy                           0.83      2500
   macro avg       0.75      0.79      0.77      2500
weighted avg       0.85      0.83      0.84      2500



In [36]:
print(df['churn'].value_counts())

churn
0    7963
1    2037
Name: count, dtype: int64
